# 06. Forma del crecimiento (segunda representación)

**Pregunta de partida.** El clustering sobre el nivel (notebooks 04 y 05) separa a los artistas por cuánto crecen: la recta de crecimiento explica el 95% de cada serie y tapa la forma. Entonces, ¿cómo crecen? ¿Hay artistas con explosiones y otros con crecimiento parejo?

**Cambio de representación.** En lugar del nivel se usa la **tasa de crecimiento mensual** (derivada FIR sobre log1p suavizado, notebook 03). La pendiente pasa a ser un valor de fondo y quedan a la vista las aceleraciones, las explosiones y la desaceleración. Una explosión, que en el nivel de una audiencia acumulada es un escalón, en la tasa es un pico.

Etapa 1 (esta versión): qué variables entran y matrices de distancia euclídea y DTW.

In [ ]:
# PAQUETES

library(data.table)
library(dtwclust)
library(ggplot2)

# dtwclust cambia el generador aleatorio al cargarse: se fija el
# generador por defecto de R para que las semillas sean reproducibles
RNGkind("Mersenne-Twister", "Inversion", "Rejection")

## 1. Configuración y carga

In [ ]:
# ============================================================
# CONFIGURACION
# ============================================================

archivos_panel_preprocesado <- list.files(
  pattern = "^artistas_ST_[0-9]+de[0-9]+_top[0-9]+_22_26_preprocesado_.+_long[.]csv$"
)
stopifnot("No hay panel preprocesado: correr el notebook 03" = length(archivos_panel_preprocesado) > 0)
ARCHIVO_PANEL_PREPROCESADO <- archivos_panel_preprocesado[which.max(file.mtime(archivos_panel_preprocesado))]

VARIABLES_PANEL <- c(
  spotify_followers = "Seguidores Spotify",
  instagram_followers = "Seguidores Instagram",
  youtube_channel_subscribers = "Suscriptores YouTube",
  spotify_playlist_count = "Playlists Spotify",
  youtube_playlist_count = "Playlists YouTube"
)

# Variables de la segunda representación: las tres audiencias
# acumuladas (decisión fundamentada en la sección 2)
VARIABLES_FORMA <- c("spotify_followers", "instagram_followers", "youtube_channel_subscribers")

COLUMNA_SERIE <- "crecimiento_mensual"

# Sin los 2 meses de cada borde: la derivada y la media móvil usan
# un solo lado (misma decisión que en las features del notebook 03)
MESES_BORDE_SIN_EXTREMOS <- 2L

# DTW: misma configuración que el notebook 04
VENTANA_DTW_MESES <- 6L
PATRON_PASOS_DTW <- dtw::symmetric1
NORMA_DTW <- "L2"

ARCHIVO_DISTANCIAS_FORMA <- file.path("chartmetric_pilotos", "distancias_forma_crecimiento.rds")
ARCHIVO_DISTANCIAS_NIVEL <- file.path("chartmetric_pilotos", "distancias_clustering.rds")
FIGURES_DIR <- "figuras"

ARCHIVO_PANEL_PREPROCESADO

In [ ]:
# ============================================================
# CARGA DEL CRECIMIENTO MENSUAL, sin los meses de borde
# ============================================================

panel <- fread(ARCHIVO_PANEL_PREPROCESADO, select = c("artist_id", "artist_name", "variable", "month", COLUMNA_SERIE))
setnames(panel, COLUMNA_SERIE, "crecimiento")
panel[, month := as.Date(month)]

MESES <- sort(unique(panel$month))
MESES_INTERIOR <- MESES[(MESES_BORDE_SIN_EXTREMOS + 1):(length(MESES) - MESES_BORDE_SIN_EXTREMOS)]
panel <- panel[month %in% MESES_INTERIOR]

artistas <- unique(panel[, .(artist_id, artist_name)])
setorder(artistas, artist_id)

cat("Artistas:", nrow(artistas), "| meses:", length(MESES_INTERIOR),
    "(", format(min(MESES_INTERIOR)), "a", format(max(MESES_INTERIOR)), ")\n")

## 2. ¿Qué variables entran?

**Pregunta.** La derivada es un pasa-altos: amplifica el ruido (clase 08). ¿Todas las variables tienen una tasa de crecimiento con señal?

La tabla mide, por variable, el crecimiento mensual típico, la proporción de meses sin crecimiento y cuánto se parece la curva de cada artista a la curva común de todos (correlación con la mediana mensual del panel). Una variable cuya tasa es casi siempre cero, o que se mueve igual para todos, no distingue formas.

In [ ]:
# ============================================================
# DIAGNOSTICO DE LA TASA DE CRECIMIENTO POR VARIABLE
# - crecimiento_tipico_pct: mediana del crecimiento mensual;
# - pct_meses_sin_crecimiento: meses con tasa exactamente 0;
# - correlacion_con_curva_comun: mediana entre artistas de la
#   correlación de su tasa con la mediana mensual del panel
#   (alta = todos se mueven igual, la variable no distingue).
# ============================================================

curva_comun <- panel[, .(mediana_mensual = median(crecimiento)), by = .(variable, month)]
panel_con_comun <- merge(panel, curva_comun, by = c("variable", "month"))

diagnostico_variables <- panel_con_comun[
  ,
  .(
    tipico = median(crecimiento),
    sin_crecimiento = mean(crecimiento == 0),
    correlacion = suppressWarnings(cor(crecimiento, mediana_mensual))
  ),
  by = .(artist_id, variable)
][
  ,
  .(
    crecimiento_tipico_pct = round(100 * median(tipico), 2),
    pct_meses_sin_crecimiento = round(100 * mean(sin_crecimiento), 1),
    correlacion_con_curva_comun = round(median(correlacion, na.rm = TRUE), 2)
  ),
  by = variable
]

diagnostico_variables[, entra := variable %in% VARIABLES_FORMA]
diagnostico_variables[order(-entra, variable)]

**Decisión.** Entran las tres audiencias acumuladas: seguidores de Spotify, seguidores de Instagram y suscriptores de YouTube. Quedan afuera las dos variables de playlists:

- **Playlists de Spotify:** la tasa de cada artista se parece mucho a la curva común (correlación mediana de 0,91): se mueven en olas compartidas, que reflejan a la plataforma más que al artista.
- **Playlists de YouTube:** crecimiento típico casi nulo (0,09% mensual) y la menor correlación con la curva común (0,31): cuentas chicas que saltan de a una playlist, mayormente ruido (17% de energía rápida, notebook 03).

La pregunta pasa a ser cómo crece la **audiencia**.

In [ ]:
# ============================================================
# SERIES: una matriz de 45 meses x 3 variables por artista
# ============================================================

ancho <- dcast(panel[variable %in% VARIABLES_FORMA], artist_id + month ~ variable, value.var = "crecimiento")
filas_por_artista <- split(seq_len(nrow(ancho)), ancho$artist_id)
series_crecimiento <- lapply(filas_por_artista, function(filas) as.matrix(ancho[filas, ..VARIABLES_FORMA]))
series_crecimiento <- series_crecimiento[as.character(artistas$artist_id)]

# Sanity check: 45 x 3 por artista, sin NA
stopifnot(
  length(series_crecimiento) == nrow(artistas),
  all(vapply(series_crecimiento, function(m) identical(dim(m), c(length(MESES_INTERIOR), length(VARIABLES_FORMA))), logical(1))),
  !anyNA(unlist(series_crecimiento))
)

cat("Series:", length(series_crecimiento), "de", length(MESES_INTERIOR), "meses x", length(VARIABLES_FORMA), "variables\n")

## 3. Distancias: ¿ahora DTW ve algo que la euclídea no ve?

Con el nivel, euclídea y DTW casi coincidían (Spearman 0,998), porque las series crecían parejo. En la tasa, una explosión es un pico: si dos artistas tienen la misma explosión con unos meses de diferencia, la euclídea los separa y DTW (ventana de 6 meses) los alinea. Misma configuración de DTW que el notebook 04.

In [ ]:
# ============================================================
# MATRICES EUCLIDEA Y DTW SOBRE LA TASA DE CRECIMIENTO
# ============================================================

nombres_artistas <- names(series_crecimiento)

matriz_euclidea_forma <- as.matrix(dist(t(vapply(series_crecimiento, as.vector, numeric(length(series_crecimiento[[1]]))))))
matriz_dtw_forma <- as.matrix(proxy::dist(series_crecimiento, method = "dtw_basic", window.size = VENTANA_DTW_MESES,
                                          norm = NORMA_DTW, step.pattern = PATRON_PASOS_DTW))
dimnames(matriz_euclidea_forma) <- dimnames(matriz_dtw_forma) <- list(nombres_artistas, nombres_artistas)

# Sanity check: simétricas, diagonal 0, sin NA, y DTW <= euclídea
stopifnot(
  isSymmetric(unname(matriz_dtw_forma), tol = 1e-8),
  max(abs(diag(matriz_dtw_forma))) < 1e-8,
  !anyNA(matriz_dtw_forma),
  all(matriz_dtw_forma <= matriz_euclidea_forma + 1e-8)
)

In [ ]:
# ============================================================
# EUCLIDEA VS DTW: nivel (notebook 04) frente a forma del crecimiento
# ============================================================

triangulo <- function(m) m[upper.tri(m)]

vecinos_compartidos <- function(m1, m2, k = 10) {
  vecinos <- function(m) apply(m, 1, function(fila) order(fila)[2:(k + 1)])
  v1 <- vecinos(m1); v2 <- vecinos(m2)
  median(vapply(seq_len(ncol(v1)), function(i) length(intersect(v1[, i], v2[, i])), integer(1)))
}

resumen_euclidea_dtw <- function(m_euclidea, m_dtw) {
  cociente <- triangulo(m_dtw) / triangulo(m_euclidea)
  data.table(
    spearman = round(cor(triangulo(m_euclidea), triangulo(m_dtw), method = "spearman"), 3),
    vecinos_compartidos_de_10 = vecinos_compartidos(m_euclidea, m_dtw),
    cociente_mediano = round(median(cociente), 3),
    pct_pares_dtw_acerca_mas_de_18 = round(100 * mean(cociente < 0.82), 1)
  )
}

distancias_nivel <- readRDS(ARCHIVO_DISTANCIAS_NIVEL)$matrices
stopifnot(identical(rownames(distancias_nivel$dtw), nombres_artistas))

comparacion_representaciones <- rbind(
  data.table(representacion = "nivel (log1p centrado)", resumen_euclidea_dtw(distancias_nivel$euclidea, distancias_nivel$dtw)),
  data.table(representacion = "forma (tasa de crecimiento)", resumen_euclidea_dtw(matriz_euclidea_forma, matriz_dtw_forma))
)

comparacion_representaciones

In [ ]:
# ============================================================
# FIGURA: cociente DTW / euclídea en las dos representaciones
# Cerca de 1, DTW no cambia nada; más abajo, DTW encuentra a los
# dos artistas más parecidos al alinear eventos corridos.
# ============================================================

datos_cociente <- rbind(
  data.table(representacion = "Nivel", cociente = triangulo(distancias_nivel$dtw) / triangulo(distancias_nivel$euclidea)),
  data.table(representacion = "Tasa de crecimiento", cociente = triangulo(matriz_dtw_forma) / triangulo(matriz_euclidea_forma))
)

grafico_cociente <- ggplot(datos_cociente, aes(x = cociente, fill = representacion)) +
  geom_density(alpha = 0.6, colour = NA, adjust = 1.2) +
  geom_vline(xintercept = 1, colour = "#1D1E2C", linetype = "dashed") +
  scale_fill_manual(values = c("Nivel" = "#B8B8B8", "Tasa de crecimiento" = "#BC4B51"), name = NULL) +
  labs(
    title = "Donde DTW empieza a importar",
    subtitle = "Cociente DTW / euclídea para cada par de artistas (1 = DTW no cambia la distancia)",
    x = "DTW / euclídea", y = "Densidad"
  ) +
  theme_minimal(base_size = 13) +
  theme(legend.position = "top", panel.grid.minor = element_blank(),
        plot.background = element_rect(fill = "white", colour = NA))

print(grafico_cociente)

ggsave(file.path(FIGURES_DIR, "forma_01_cociente_dtw_euclidea.png"), grafico_cociente,
       width = 10, height = 5.5, dpi = 300, bg = "white")

In [ ]:
# ============================================================
# GUARDADO (archivo local, excluido de git)
# ============================================================

saveRDS(
  list(
    fecha_generacion = Sys.time(),
    archivo_panel = ARCHIVO_PANEL_PREPROCESADO,
    columna_serie = COLUMNA_SERIE,
    variables = VARIABLES_FORMA,
    meses = MESES_INTERIOR,
    ventana_dtw_meses = VENTANA_DTW_MESES,
    artistas = artistas,
    matrices = list(euclidea = matriz_euclidea_forma, dtw = matriz_dtw_forma)
  ),
  ARCHIVO_DISTANCIAS_FORMA
)

cat("Matrices guardadas en", ARCHIVO_DISTANCIAS_FORMA, "\n")

## Lectura de la etapa 1

- **DTW importa más que con el nivel, pero no da vuelta el panorama.** El orden de los pares sigue muy parecido (Spearman 0,996 frente a 0,998), aunque los vecindarios cambian más: cada artista comparte 6 de sus 10 vecinos más cercanos entre euclídea y DTW (8 con el nivel), y DTW acerca más de un 18% a 4 de cada 100 pares (1 de cada 100 con el nivel).
- **Lectura:** los corrimientos en el tiempo existen, pero afectan a una minoría de artistas. DTW es la distancia que los respeta; el clustering (etapa 2) dirá si esa minoría forma un perfil propio.